# PlantDoc AI - Training MobileNetV2 di Google Colab

Klasifikasi 4 kondisi daun tomat: **Healthy, Early Blight, Late Blight, Leaf Mold**.

Pipeline ini mempertahankan arsitektur MobileNetV2 dan preprocessing yang sama, tetapi menambahkan provenance dataset, deduplikasi SHA-256, group-aware split, class weight, checkpoint berbasis `val_loss`, penyimpanan metrik, dan validasi paritas TFLite.

**Sebelum mulai:**
1. Pilih runtime T4 GPU bila tersedia.
2. Jalankan sel dari atas secara berurutan.
3. Untuk Kaggle: buat secret baru setelah revoke token lama, lalu tambahkan `KAGGLE_USERNAME` dan `KAGGLE_KEY` di Colab Secrets.
4. Jangan menulis token langsung ke notebook atau Git.
5. Jika Colab meminta restart setelah instalasi paket, restart runtime lalu lanjutkan dari sel berikutnya.

**Batasan:** model ini belum memiliki kelas `unknown`. Deteksi non-daun pada web masih berupa filter heuristik, sehingga notebook ini tidak mengklaim kemampuan open-set rejection.

In [ ]:
!nvidia-smi
!pip -q install "tensorflow==2.21.0" "keras==3.15.1" "scikit-learn==1.9.1" "matplotlib==3.11.2" "kagglehub==1.0.3" "datasets==5.2.0" "tensorflow-datasets==4.9.10" "pillow==13.0.0" "numpy==2.5.5" "ai-edge-litert==2.4.0"

## 1. Siapkan dataset dan split

Sumber dicoba berurutan: Kaggle, Hugging Face, lalu TensorFlow Datasets. Setiap percobaan dimulai dari direktori bersih sehingga data parsial dari sumber sebelumnya tidak tercampur.

Sebelum split, gambar duplikat identik dibuang berdasarkan SHA-256. Split 70/15/15 dilakukan per kelas dengan `GroupShuffleSplit`; group berasal dari nama file sumber atau ID baris dataset. Rasio aktual dapat sedikit berbeda karena ukuran group tidak seragam.

Output tahap ini:
- `dataset_manifest.json`: sumber, versi paket, jumlah kelas, duplikat, dan jumlah tiap split.
- `split_manifest.csv`: asal setiap gambar, group, hash, kelas, dan split.

In [ ]:
import csv
import hashlib
import json
import platform
import shutil
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import kagglehub
import tensorflow_datasets as tfds
from datasets import load_dataset
from google.colab import userdata
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
SELECTED = {
    "healthy": "Tomato___healthy",
    "early": "Tomato___Early_blight",
    "late": "Tomato___Late_blight",
    "mold": "Tomato___Leaf_Mold",
}
HF_LABELS = {
    "healthy": "Tomato healthy",
    "early": "Tomato Early blight",
    "late": "Tomato Late blight",
    "mold": "Tomato Leaf Mold",
}
ROOT = Path("/content/plantdoc_data")


def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)

sources = {key: [] for key in SELECTED}
seen_hashes = {}
stats = {key: {"loaded": 0, "duplicates": 0} for key in SELECTED}


def reset_sources():
    global sources, seen_hashes, stats
    shutil.rmtree(ROOT / "all", ignore_errors=True)
    sources = {key: [] for key in SELECTED}
    seen_hashes = {}
    stats = {key: {"loaded": 0, "duplicates": 0} for key in SELECTED}


def next_path(web_key, suffix):
    path = ROOT / "all" / web_key / f"{web_key}_{len(sources[web_key]):06d}{suffix.lower()}"
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def register_file(path, web_key, group_id):
    path = Path(path)
    digest = sha256_file(path)
    if digest in seen_hashes:
        if seen_hashes[digest] != web_key:
            raise ValueError(f"Hash identik dengan label berbeda: {path}")
        stats[web_key]["duplicates"] += 1
        path.unlink(missing_ok=True)
        return
    seen_hashes[digest] = web_key
    sources[web_key].append({
        "path": path,
        "group_id": str(group_id),
        "sha256": digest,
    })
    stats[web_key]["loaded"] += 1


def copy_source(src, web_key, group_id):
    dst = next_path(web_key, Path(src).suffix)
    shutil.copy2(src, dst)
    register_file(dst, web_key, group_id)


def save_image(image, web_key, group_id):
    dst = next_path(web_key, ".jpg")
    image.convert("RGB").save(dst, format="JPEG", quality=95)
    register_file(dst, web_key, group_id)


In [ ]:
def load_kaggle():
    username = userdata.get("KAGGLE_USERNAME")
    kaggle_key = userdata.get("KAGGLE_KEY")
    if not username or not kaggle_key:
        raise RuntimeError("Kaggle secret belum tersedia")
    dataset_id = "soumiknafiul/plantvillage-dataset-labeled"
    base = Path(kagglehub.dataset_download(dataset_id))
    variants = {
        "healthy": [("tomato", "healthy")],
        "early": [("tomato", "early", "blight")],
        "late": [("tomato", "late", "blight")],
        "mold": [("tomato", "leaf", "mold")],
    }
    extensions = {".jpg", ".jpeg", ".png"}
    for web_key, parts in variants.items():
        for folder in base.rglob("*"):
            if not folder.is_dir():
                continue
            name = folder.name.lower()
            if not all(part in name for part in parts):
                continue
            for src in folder.rglob("*"):
                if src.is_file() and src.suffix.lower() in extensions:
                    copy_source(src, web_key, src.stem)
    return dataset_id, None


def load_huggingface():
    dataset_id = "manoela/plantvillage"
    dataset = load_dataset(dataset_id, split="train", streaming=True)
    for index, row in enumerate(dataset):
        web_key = next((key for key, label in HF_LABELS.items() if row["caption"] == label), None)
        if web_key is not None:
            save_image(row["image"], web_key, f"hf-{index}")
    revision = getattr(getattr(dataset, "info", None), "sha", None)
    return dataset_id, revision


def load_tfds():
    dataset_id = "tensorflow_datasets/plant_village"
    dataset, info = tfds.load("plant_village", split="train", as_supervised=True, with_info=True)
    class_names_full = info.features["label"].names
    selected_ids = {class_names_full.index(name): key for key, name in SELECTED.items()}
    for index, (image, label_id) in enumerate(tfds.as_numpy(dataset)):
        web_key = selected_ids.get(int(label_id))
        if web_key is not None:
            save_image(Image.fromarray(image), web_key, f"tfds-{index}")
    return dataset_id, str(info.version)


In [ ]:
loaders = [
    ("Kaggle", load_kaggle),
    ("Hugging Face", load_huggingface),
    ("TensorFlow Datasets", load_tfds),
]
fallback_errors = []
for source_name, loader in loaders:
    reset_sources()
    try:
        source_dataset, source_revision = loader()
        missing = [key for key in SELECTED if not sources[key]]
        if missing:
            raise RuntimeError(f"Kelas tidak ditemukan: {missing}")
        SOURCE = source_name
        SOURCE_DATASET = source_dataset
        SOURCE_REVISION = source_revision
        break
    except Exception as error:
        fallback_errors.append({"source": source_name, "error": str(error)})
        print(f"Fallback setelah {source_name}: {error}")
else:
    raise RuntimeError("Semua sumber dataset gagal; lihat fallback_errors")

split_counts = {}
split_rows = []
for web_key, records in sources.items():
    if len(records) < 6:
        raise RuntimeError(f"Kelas {web_key} hanya memiliki {len(records)} citra unik")
    paths = [record["path"] for record in records]
    groups = [record["group_id"] for record in records]
    labels = [web_key] * len(records)
    train_idx, rest_idx = next(
        GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=SEED).split(paths, labels, groups)
    )
    rest_paths = [paths[index] for index in rest_idx]
    rest_groups = [groups[index] for index in rest_idx]
    rest_labels = [labels[index] for index in rest_idx]
    val_idx, test_idx = next(
        GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=SEED + 1).split(
            rest_paths, rest_labels, rest_groups
        )
    )
    splits = {
        "train": [records[index] for index in train_idx],
        "val": [records[rest_idx[index]] for index in val_idx],
        "test": [records[rest_idx[index]] for index in test_idx],
    }
    group_sets = {
        split: {record["group_id"] for record in split_records}
        for split, split_records in splits.items()
    }
    assert group_sets["train"].isdisjoint(group_sets["val"])
    assert group_sets["train"].isdisjoint(group_sets["test"])
    assert group_sets["val"].isdisjoint(group_sets["test"])
    split_counts[web_key] = {}
    for split, split_records in splits.items():
        destination = ROOT / split / web_key
        destination.mkdir(parents=True, exist_ok=True)
        for record in split_records:
            src = record["path"]
            output_path = destination / src.name
            shutil.copy2(src, output_path)
            split_rows.append({
                "split": split,
                "class": web_key,
                "group_id": record["group_id"],
                "sha256": record["sha256"],
                "source_path": str(src.relative_to(ROOT)),
                "path": str(output_path.relative_to(ROOT)),
            })
        split_counts[web_key][split] = len(split_records)

with (ROOT / "split_manifest.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["split", "class", "group_id", "sha256", "source_path", "path"],
    )
    writer.writeheader()
    writer.writerows(split_rows)

manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "source": SOURCE,
    "dataset_id": SOURCE_DATASET,
    "revision": SOURCE_REVISION,
    "selected_classes": SELECTED,
    "deduplication": "sha256",
    "split_strategy": "group-aware per class, target 70/15/15",
    "class_counts": split_counts,
    "duplicates_removed": {key: stats[key]["duplicates"] for key in SELECTED},
    "fallback_errors": fallback_errors,
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "tensorflow": package_version("tensorflow"),
        "keras": package_version("keras"),
        "scikit_learn": package_version("scikit-learn"),
        "matplotlib": package_version("matplotlib"),
        "kagglehub": package_version("kagglehub"),
        "datasets": package_version("datasets"),
        "tensorflow_datasets": package_version("tensorflow-datasets"),
        "pillow": package_version("pillow"),
        "numpy": package_version("numpy"),
        "ai_edge_litert": package_version("ai-edge-litert"),
    },
}
(ROOT / "dataset_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("Sumber dataset:", SOURCE, SOURCE_DATASET, SOURCE_REVISION)
for web_key in SELECTED:
    print(
        f"{web_key:8s} train={split_counts[web_key]['train']} "
        f"val={split_counts[web_key]['val']} test={split_counts[web_key]['test']} "
        f"duplikat={stats[web_key]['duplicates']}"
    )


## 2. Verifikasi dataset dan split

Notebook memastikan setiap kelas tersedia di train, validation, dan test. Pemeriksaan berikut juga membaca ulang manifest untuk memastikan tidak ada group sumber yang muncul di lebih dari satu split.

In [ ]:
for split in ("train", "val", "test"):
    counts = {key: len(list((ROOT / split / key).glob("*"))) for key in SELECTED}
    assert all(counts.values()), f"Split {split} memiliki kelas kosong: {counts}"
    print(f"{split:5s}: {counts}")

with (ROOT / "split_manifest.csv").open(encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))
assert len({row["sha256"] for row in rows}) == len(rows), "Ada hash duplikat setelah split"
for web_key in SELECTED:
    groups_by_split = {
        split: {row["group_id"] for row in rows if row["class"] == web_key and row["split"] == split}
        for split in ("train", "val", "test")
    }
    assert groups_by_split["train"].isdisjoint(groups_by_split["val"])
    assert groups_by_split["train"].isdisjoint(groups_by_split["test"])
    assert groups_by_split["val"].isdisjoint(groups_by_split["test"])
print("Verifikasi deduplikasi dan group split: OK")

## 3. Siapkan pipeline data dan class weight

Dataset dimuat tanpa augmentasi di luar model. Class weight dihitung dari jumlah citra train tiap kelas.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)
IMG_SIZE = (224, 224)
BATCH = 32
EPOCHS_HEAD = 8
EPOCHS_FT = 8

train_ds = tf.keras.utils.image_dataset_from_directory(
    ROOT / "train", image_size=IMG_SIZE, batch_size=BATCH, shuffle=True, seed=42
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    ROOT / "val", image_size=IMG_SIZE, batch_size=BATCH, shuffle=False
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    ROOT / "test", image_size=IMG_SIZE, batch_size=BATCH, shuffle=False
)

class_names = train_ds.class_names
print("Urutan kelas model:", class_names)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


def infer_web_key(folder):
    name = folder.lower()
    if "healthy" in name:
        return "healthy"
    if "early" in name:
        return "early"
    if "mold" in name:
        return "mold"
    if "late" in name:
        return "late"
    return folder


web_keys = [infer_web_key(name) for name in class_names]
assert set(web_keys) == set(SELECTED), f"Peta kelas web tidak lengkap: {web_keys}"
train_counts = {
    index: len(list((ROOT / "train" / name).glob("*")))
    for index, name in enumerate(class_names)
}
total_train = sum(train_counts.values())
class_weight = {
    index: total_train / (len(class_names) * count)
    for index, count in train_counts.items()
}
print("Peta ke web:", dict(zip(class_names, web_keys)))
print("Jumlah train:", train_counts)
print("Class weight:", class_weight)


## 4. Bangun model MobileNetV2 (transfer learning)

Arsitektur dan preprocessing tidak diubah: input 224×224, augmentasi ringan di dalam model, `mobilenet_v2.preprocess_input`, backbone ImageNet, GlobalAveragePooling, Dropout 0.25, dan softmax empat kelas.

Backbone dibekukan pada tahap awal. `backbone(x, training=False)` dipertahankan agar statistik BatchNormalization tidak berubah selama fine-tuning; perilaku ini kini dicatat sebagai keputusan eksperimen.

`image_dataset_from_directory` mengurutkan kelas secara alfabetis. Peta `class_names → web_keys` disimpan saat ekspor agar urutan prediksi web tetap benar.

In [ ]:
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.06),
    layers.RandomZoom(0.08),
], name="augment")

backbone = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet")
backbone.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = augment(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = backbone(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])
model.summary()


## 5. Latih

Tahap A hanya melatih classification head. Tahap B membuka 30 lapisan terakhir backbone dengan learning rate kecil. Kedua tahap memakai class weight untuk mengurangi pengaruh ketidakseimbangan kelas.

Checkpoint dan early stopping memantau `val_loss`, bukan akurasi validasi. Model terbaik dipilih dari nilai `val_loss` terendah di kedua tahap, lalu disimpan sebagai `best.keras`. History training dan konfigurasi eksperimen ikut disimpan agar hasil dapat diaudit.

In [ ]:
import json
import shutil
from pathlib import Path

import numpy as np

OUT = Path("/content/plantdoc_model")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)

for name in ("dataset_manifest.json", "split_manifest.csv"):
    shutil.copy2(ROOT / name, OUT / name)


def training_callbacks(checkpoint_name):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            str(OUT / checkpoint_name),
            monitor="val_loss",
            save_best_only=True,
            mode="min",
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=3,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.3,
            patience=2,
            min_lr=1e-7,
        ),
    ]


print("=== Tahap A: classification head ===")
history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    class_weight=class_weight,
    callbacks=training_callbacks("best_head.keras"),
)

print("=== Tahap B: fine-tuning 30 lapisan terakhir ===")
backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT,
    class_weight=class_weight,
    callbacks=training_callbacks("best_finetune.keras"),
)

head_best_epoch = int(np.argmin(history_head.history["val_loss"]))
ft_best_epoch = int(np.argmin(history_ft.history["val_loss"]))
head_best_loss = float(history_head.history["val_loss"][head_best_epoch])
ft_best_loss = float(history_ft.history["val_loss"][ft_best_epoch])
if head_best_loss <= ft_best_loss:
    selected_stage = "head"
    selected_epoch = head_best_epoch + 1
    selected_checkpoint = OUT / "best_head.keras"
else:
    selected_stage = "fine_tune"
    selected_epoch = ft_best_epoch + 1
    selected_checkpoint = OUT / "best_finetune.keras"
selected_val_loss = min(head_best_loss, ft_best_loss)

shutil.copy2(selected_checkpoint, OUT / "best.keras")
(OUT / "training_history.json").write_text(
    json.dumps(
        {
            "head": history_head.history,
            "fine_tune": history_ft.history,
            "selected_stage": selected_stage,
            "selected_epoch": selected_epoch,
            "selected_val_loss": selected_val_loss,
        },
        indent=2,
    ),
    encoding="utf-8",
)
(OUT / "training_config.json").write_text(
    json.dumps(
        {
            "seed": 42,
            "input_size": list(IMG_SIZE),
            "batch_size": BATCH,
            "epochs_head": EPOCHS_HEAD,
            "epochs_fine_tune": EPOCHS_FT,
            "fine_tune_layers": 30,
            "monitor": "val_loss",
            "class_weight": class_weight,
        },
        indent=2,
    ),
    encoding="utf-8",
)
(OUT / "best_head.keras").unlink()
(OUT / "best_finetune.keras").unlink()
print(f"Checkpoint dipilih: {selected_stage}, epoch {selected_epoch}, val_loss={selected_val_loss:.6f}")

## 6. Evaluasi pada data test

Data test tidak dipakai saat training, class weighting, checkpoint, maupun early stopping. Evaluasi menyimpan accuracy, loss, precision, recall, F1 per kelas, macro-F1, confusion matrix, dan prediksi setiap citra test. Angka inilah yang boleh dilaporkan sebagai akurasi model, bukan akurasi training atau smoke test beberapa gambar.

In [ ]:
import csv
from datetime import datetime, timezone

import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

best = tf.keras.models.load_model(OUT / "best.keras")
assert test_ds.class_names == class_names, "Urutan kelas test berbeda dari training"
loss, accuracy = best.evaluate(test_ds, verbose=0)

y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
y_prob = best.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
report = classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))

evaluation = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "split": "test",
    "samples": int(len(y_true)),
    "classes": class_names,
    "web_keys": web_keys,
    "loss": float(loss),
    "accuracy": float(accuracy),
    "macro_precision": float(report["macro avg"]["precision"]),
    "macro_recall": float(report["macro avg"]["recall"]),
    "macro_f1": float(report["macro avg"]["f1-score"]),
    "selected_stage": selected_stage,
    "selected_epoch": selected_epoch,
    "selected_val_loss": float(selected_val_loss),
}
(OUT / "evaluation.json").write_text(
    json.dumps(evaluation, indent=2, ensure_ascii=False), encoding="utf-8"
)
(OUT / "classification_report.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
)

with (OUT / "confusion_matrix.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["actual"] + class_names)
    for name, row in zip(class_names, cm):
        writer.writerow([name, *row.tolist()])

with (OUT / "predictions.csv").open("w", newline="", encoding="utf-8") as handle:
    fieldnames = ["path", "true_class", "predicted_class", *class_names]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    for index, (true_index, pred_index, probabilities) in enumerate(zip(y_true, y_pred, y_prob)):
        path = Path(test_ds.file_paths[index])
        writer.writerow({
            "path": str(path.relative_to(ROOT)),
            "true_class": class_names[int(true_index)],
            "predicted_class": class_names[int(pred_index)],
            **{
                name: float(probability)
                for name, probability in zip(class_names, probabilities)
            },
        })

print(f"Test loss: {loss:.6f}")
print(f"Test accuracy: {accuracy*100:.2f}%")
print(f"Macro F1: {report['macro avg']['f1-score']:.4f}")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Greens", colorbar=False)
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.savefig(OUT / "confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()

## 7. Ekspor model dan validasi TFLite

`labels.json` menyimpan urutan kelas model dan kunci web. Sebelum model dipakai, output Keras dan LiteRT dibandingkan pada seluruh data test dengan toleransi `1e-4`; notebook akan berhenti jika paritas gagal.

Output juga mencakup metadata input/output, hasil benchmark inferensi, dan hash setiap artefak. Hash digunakan untuk memastikan file yang disalin ke web atau Drive sama dengan hasil ekspor.

In [ ]:
from datetime import datetime, timezone

from ai_edge_litert.interpreter import Interpreter

converter = tf.lite.TFLiteConverter.from_keras_model(best)
tflite_model = converter.convert()
(OUT / "model.tflite").write_bytes(tflite_model)

labels = {"classes": class_names, "web_keys": web_keys}
(OUT / "labels.json").write_text(
    json.dumps(labels, indent=2, ensure_ascii=False), encoding="utf-8"
)

interpreter = Interpreter(model_path=str(OUT / "model.tflite"))
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
assert len(input_details) == 1 and len(output_details) == 1
input_index = input_details[0]["index"]
output_index = output_details[0]["index"]
assert input_details[0]["shape"].tolist() == [1, 224, 224, 3]
assert input_details[0]["dtype"] == np.float32
assert output_details[0]["shape"].tolist() == [1, len(class_names)]

atol = 1e-4
rtol = 1e-4
sample_count = 0
top1_matches = 0
allclose_matches = 0
max_abs_diff = 0.0
sum_abs_diff = 0.0
for images, _ in test_ds:
    keras_probabilities = best(images, training=False).numpy()
    for image, keras_probabilities_one in zip(images.numpy(), keras_probabilities):
        interpreter.set_tensor(input_index, np.expand_dims(image, axis=0))
        interpreter.invoke()
        lite_probabilities = interpreter.get_tensor(output_index)[0]
        difference = np.abs(keras_probabilities_one - lite_probabilities)
        sample_count += 1
        top1_matches += int(
            np.argmax(keras_probabilities_one) == np.argmax(lite_probabilities)
        )
        allclose_matches += int(
            np.allclose(keras_probabilities_one, lite_probabilities, atol=atol, rtol=rtol)
        )
        max_abs_diff = max(max_abs_diff, float(difference.max()))
        sum_abs_diff += float(difference.sum())

assert sample_count == len(y_true), "Paritas tidak mencakup seluruh data test"
parity = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "runtime": "ai-edge-litert",
    "runtime_version": package_version("ai-edge-litert"),
    "samples": sample_count,
    "atol": atol,
    "rtol": rtol,
    "allclose_passed": allclose_matches == sample_count,
    "top1_agreement": top1_matches / sample_count,
    "max_absolute_difference": max_abs_diff,
    "mean_absolute_difference": sum_abs_diff / (sample_count * len(class_names)),
}
assert parity["allclose_passed"], f"Paritas Keras dan LiteRT gagal: {parity}"
assert parity["top1_agreement"] == 1.0, f"Prediksi top-1 berbeda: {parity}"
(OUT / "tflite_parity.json").write_text(
    json.dumps(parity, indent=2), encoding="utf-8"
)

In [ ]:
import time

benchmark_input = next(iter(test_ds))[0][:1].numpy()
for _ in range(5):
    interpreter.set_tensor(input_index, benchmark_input)
    interpreter.invoke()
latencies_ms = []
for _ in range(30):
    started = time.perf_counter()
    interpreter.set_tensor(input_index, benchmark_input)
    interpreter.invoke()
    latencies_ms.append((time.perf_counter() - started) * 1000)
parity["benchmark"] = {
    "iterations": len(latencies_ms),
    "warmup_iterations": 5,
    "median_ms": float(np.median(latencies_ms)),
    "p95_ms": float(np.quantile(latencies_ms, 0.95)),
    "scope": "set_tensor dan invoke; tidak termasuk decode, resize, atau I/O",
}
(OUT / "tflite_parity.json").write_text(
    json.dumps(parity, indent=2), encoding="utf-8"
)

dataset_manifest = json.loads((OUT / "dataset_manifest.json").read_text(encoding="utf-8"))
metadata = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model": "MobileNetV2",
    "input": {
        "shape": input_details[0]["shape"].tolist(),
        "dtype": np.dtype(input_details[0]["dtype"]).name,
    },
    "output": {
        "shape": output_details[0]["shape"].tolist(),
        "dtype": np.dtype(output_details[0]["dtype"]).name,
    },
    "classes": class_names,
    "web_keys": web_keys,
    "preprocessing": "mobilenet_v2.preprocess_input tersemat di model",
    "dataset_id": dataset_manifest["dataset_id"],
    "dataset_revision": dataset_manifest["revision"],
    "split_seed": dataset_manifest["seed"],
    "open_set_rejection": False,
}
(OUT / "model_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)

artifact_manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "artifacts": {
        path.name: {
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in sorted(OUT.iterdir())
        if path.is_file() and path.name != "artifact_manifest.json"
    },
}
(OUT / "artifact_manifest.json").write_text(
    json.dumps(artifact_manifest, indent=2), encoding="utf-8"
)

print("Paritas Keras ↔ LiteRT:", parity)
print("File di", OUT)
for path in sorted(OUT.glob("*")):
    print(f" - {path.name} ({path.stat().st_size/1024/1024:.2f} MB)")
print(json.dumps(labels, indent=2, ensure_ascii=False))

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DEST = Path("/content/drive/MyDrive/plantdoc")
DEST.mkdir(parents=True, exist_ok=True)
for path in OUT.iterdir():
    if path.is_file():
        destination = DEST / path.name
        shutil.copy2(path, destination)
        assert sha256_file(path) == sha256_file(destination), f"Hash salinan Drive tidak cocok: {path.name}"
print("Semua artefak tersimpan dan hash terverifikasi di", DEST)

## 8. Hubungkan ke web

Aplikasi di `backend/app.py` memuat `artifacts/models/model.tflite` dengan LiteRT, tanpa TensorFlow penuh. Endpoint `POST /analyze` melayani antarmuka HTML dan `POST /predict` melayani API JSON; keduanya menerima multipart field `file`.

Salin `model.tflite` dan `labels.json` hasil ekspor ke `artifacts/models/`. Urutan kelas harus mengikuti `web_keys` (`early`, `healthy`, `late`, `mold`) agar prediksi tidak tertukar. Simpan `evaluation.json`, `classification_report.json`, `tflite_parity.json`, dan `artifact_manifest.json` bersama catatan eksperimen.

Model hanya mengenali empat kondisi daun tomat dan belum memiliki kelas `unknown`. Filter non-daun di web masih heuristik, bukan jaminan penolakan semua gambar bukan tanaman.

Untuk menjalankan: `python -m pip install -r requirements.txt`, lalu `python -m backend.app`, kemudian buka `http://127.0.0.1:5000`.